# 🏴 OZZ + RAPTOR v6 — Qwen 0.5B (Fast)

**DEF CON 34 AI Village HALctf** — Modelo leve, download rápido

In [ ]:
!pip install -q fastapi uvicorn pydantic requests transformers torch accelerate semgrep
import os, sys, time, requests, subprocess, json, re
from datetime import datetime
WORK = '/kaggle/working'
REPORTS = f'{WORK}/reports'
os.makedirs(REPORTS, exist_ok=True)
os.makedirs(f'{WORK}/hf_cache', exist_ok=True)
print('✅ Setup!')

In [ ]:
!git clone --depth 1 https://github.com/Tretabolt/ozz-halctf.git {WORK}/ozz 2>&1 | tail -3
!git clone --depth 1 https://github.com/gadievron/raptor.git {WORK}/raptor 2>&1 | tail -3
OZZ = f'{WORK}/ozz'
print('✅ Repos clonados!')

In [ ]:
# Servidor Qwen 0.5B (~1GB, baixa em ~1min)
with open(f'{WORK}/qwen_server.py', 'w') as f:
    f.write('''
import os, torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Dict, Optional, Union
from transformers import AutoModelForCausalLM, AutoTokenizer
import uvicorn

app = FastAPI()
model_id = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
cache_dir = "/kaggle/working/hf_cache"
print(f"Loading {model_id}...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir, torch_dtype=dtype, device_map=device, trust_remote_code=True)
print(f"Ready on {device}!", flush=True)

class Req(BaseModel):
    model: str
    messages: List[Dict[str, str]]
    max_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.3
    stop: Optional[Union[str, List[str]]] = None

@app.get("/v1/models")
def models():
    return {"data": [{"id": model_id}]}

@app.post("/v1/chat/completions")
def chat(req: Req):
    try:
        prompt = tokenizer.apply_chat_template(req.messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(device)
        max_tok = min(req.max_tokens or 512, 512)
        temp = req.temperature if req.temperature is not None else 0.3
        gk = {"max_new_tokens": max_tok, "do_sample": temp > 0, "pad_token_id": tokenizer.pad_token_id}
        if temp > 0: gk["temperature"] = temp
        with torch.no_grad():
            out = model.generate(**inputs, **gk)
        ids = out[0][inputs.input_ids.shape[1]:]
        text = tokenizer.decode(ids, skip_special_tokens=True)
        return {"choices": [{"message": {"role": "assistant", "content": text}}]}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
''')

log = open(f'{WORK}/qwen_server.log', 'w')
proc = subprocess.Popen(['python3', '-u', f'{WORK}/qwen_server.py'],
    stdout=log, stderr=log, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
print('⚡ Qwen 0.5B server starting...')

In [ ]:
# Health check (0.5B carrega em ~30s)
print('⏳ Aguardando Qwen 0.5B...')
llm_ok = False
for i in range(24):
    try:
        r = requests.get('http://localhost:8000/v1/models', timeout=2)
        if r.status_code == 200:
            print('✅ Qwen pronto!')
            llm_ok = True
            break
    except: pass
    time.sleep(5)

if not llm_ok:
    print('❌ Falhou! Logs:')
    with open(f'{WORK}/qwen_server.log') as f:
        print(f.read()[-1500:])
else:
    try:
        r = requests.post('http://localhost:8000/v1/chat/completions',
            json={'model': 'test', 'messages': [{'role': 'user', 'content': 'Say READY'}], 'max_tokens': 10},
            timeout=30)
        d = r.json()
        if 'choices' in d:
            print(f'🧪 LLM: {d["choices"][0]["message"]["content"][:50]}')
    except Exception as e:
        print(f'⚠️ Test: {e}')

In [ ]:
# 🔬 RAPTOR — Semgrep
print('='*60)
print('🔬 RAPTOR — Semgrep Scan')
print('='*60)

targets = {'target-01-web': f'{OZZ}/universe/target-01', 'target-03-api': f'{OZZ}/universe/target-03'}
all_findings = []
for name, path in targets.items():
    if not os.path.isdir(path): continue
    print(f'\n🔍 {name}...')
    sarif = f'{REPORTS}/{name}.sarif'
    try:
        subprocess.run(['semgrep', '--config', 'auto', '--config', 'p/owasp-top-ten',
            '--config', 'p/security-audit', '--sarif', '--output', sarif, '--quiet', path],
            capture_output=True, text=True, timeout=120)
        if os.path.exists(sarif):
            with open(sarif) as f: data = json.load(f)
            for run in data.get('runs', []):
                for res in run.get('results', []):
                    rule = res.get('ruleId', '?')
                    msg = res.get('message', {}).get('text', '')
                    locs = res.get('locations', [])
                    fp = locs[0].get('physicalLocation', {}).get('artifactLocation', {}).get('uri', '') if locs else ''
                    ln = locs[0].get('physicalLocation', {}).get('region', {}).get('startLine', 0) if locs else 0
                    sev = 'medium'
                    for rr in run.get('tool', {}).get('driver', {}).get('rules', []):
                        if rr.get('id') == rule:
                            lvl = rr.get('defaultConfiguration', {}).get('level', '')
                            if 'error' in lvl: sev = 'critical'
                            elif 'warning' in lvl: sev = 'high'
                    all_findings.append({'target': name, 'rule': rule, 'severity': sev, 'message': msg, 'file': fp, 'line': ln})
                    icon = {'critical': '🔴', 'high': '🟠'}.get(sev, '🟡')
                    print(f'  {icon} {rule}: {msg[:60]}')
                print(f'  📊 {len(run.get("results",[]))} findings')
    except Exception as e:
        print(f'  ❌ {e}')

guidance = '\n'.join(f'- [{f["severity"].upper()}] {f["rule"]} at {f["file"]}:{f["line"]}' for f in all_findings)
print(f'\n✅ RAPTOR: {len(all_findings)} findings')

In [ ]:
# 🧠 Qwen gera payloads
print('='*60)
print('🧠 Qwen — Payload Generation')
print('='*60)

llm_payloads = ''
if llm_ok:
    prompt = f"""You are a CTF pentesting agent. Based on these Semgrep findings, generate exploit payloads.

Findings:
{guidance[:2000]}

For each vulnerability, provide:
1. curl command to exploit
2. Expected flag format

JSON format: [{{"vuln": "...", "payload": "curl ...", "flag": "flag{{...}}"}}]
"""
    try:
        r = requests.post('http://localhost:8000/v1/chat/completions',
            json={'model': 'test', 'messages': [{'role': 'user', 'content': prompt}],
                  'max_tokens': 1024, 'temperature': 0.2}, timeout=120)
        d = r.json()
        if 'choices' in d:
            llm_payloads = d['choices'][0]['message']['content']
            print(f'\n📝 Qwen payloads ({len(llm_payloads)} chars):')
            print(llm_payloads[:2000])
            with open(f'{REPORTS}/llm_payloads.txt', 'w') as f:
                f.write(llm_payloads)
    except Exception as e:
        print(f'❌ {e}')
else:
    print('⚠️ LLM indisponível')

In [ ]:
# 📊 Relatório Final
print('='*60)
print('📊 RELATÓRIO FINAL — OZZ + RAPTOR v6')
print('='*60)

sev = {'critical': 0, 'high': 0, 'medium': 0, 'low': 0}
for f in all_findings:
    sev[f['severity']] = sev.get(f['severity'], 0) + 1

report = {
    'pipeline': 'OZZ + RAPTOR v6',
    'timestamp': datetime.now().isoformat(),
    'model': 'Qwen2.5-Coder-0.5B-Instruct',
    'llm_ok': llm_ok,
    'findings': len(all_findings),
    'severity': sev,
    'payloads_generated': len(llm_payloads) > 0,
}
with open(f'{REPORTS}/final_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f'\n🔬 RAPTOR: {len(all_findings)} findings')
print(f'  🔴 Critical: {sev["critical"]}  🟠 High: {sev["high"]}')
print(f'🧠 LLM: {"✅ Qwen 0.5B" if llm_ok else "❌"}')
print(f'📄 {REPORTS}/final_report.json')

if 'proc' in globals(): proc.terminate()
if 'log' in globals() and not log.closed: log.close()
print('\n🏴 v6 finalizado!')